# Notebook 05 — Difficulty Dependence (Supplementary Tables 3–5)

Shows how each metacognitive measure changes as a function of task difficulty. We compare measures computed at hard vs easy trials for three datasets:
- **Shekhar 2021** — three discrete contrast levels
- **Rouault 2018 Expt 1 & 2** — median split on dot-count difference

Matching MATLAB `ana_taskPerformance.m`: per measure per difficulty level, values more than 3 SD from the mean are removed, and if any level is NaN for a subject on a given measure, all levels are set to NaN for that subject.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import sys, os, warnings
warnings.filterwarnings('ignore')

# ── locate repository root and add src/ to path ──────────────
REPO = os.path.abspath(os.path.join(
    os.getcwd(), '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'))
DATA = os.path.join(REPO, 'matlab', 'metasignal_mat', 'Preprocess', 'orig_csv_files')
OUT  = os.path.join(REPO, 'notebooks', 'precomputed')
sys.path.insert(0, os.path.join(REPO, 'src'))
os.makedirs(OUT, exist_ok=True)

import numpy as np
import pandas as pd
from scipy import stats
from metasignal.stdpy.compute_all import compute_all_measures
print("metasignal loaded successfully.")


In [ ]:
MEASURE_NAMES = [
    "meta-d'", "AUC2", "Gamma", "Phi", "DeltaConf",
    "M-Ratio", "AUC2-Ratio", "Gamma-Ratio", "Phi-Ratio", "DeltaConf-Ratio",
    "M-Diff",  "AUC2-Diff",  "Gamma-Diff",  "Phi-Diff",  "DeltaConf-Diff",
    "meta-noise", "meta-uncertainty",
    "d'", "Criterion", "Confidence",
]
N_MEAS = 20


In [ ]:
# ── t-test matching MATLAB perform_ttest.m ───────────────────
def ttest_1samp(data):
    """One-sample t-test vs 0. Returns (t, df, p, cohen_d, ci_lo, ci_hi).
    Cohen's d = t/sqrt(n) matching MATLAB: Cohen_d = t/sqrt(df+1)."""
    x = np.asarray(data, float)
    x = x[~np.isnan(x)]
    n = len(x)
    if n < 2:
        return (np.nan,)*6
    t, p = stats.ttest_1samp(x, 0)
    d    = t / np.sqrt(n)
    sem  = x.std(ddof=1) / np.sqrt(n)
    ci   = x.mean() + stats.t.ppf([0.025, 0.975], n-1) * sem
    return t, n-1, p, d, ci[0], ci[1]

def p_stars(p):
    if np.isnan(p): return ''
    if p < 0.001:   return '***'
    if p < 0.01:    return '**'
    if p < 0.05:    return '*'
    return 'ns'

# ── one-way repeated-measures ANOVA ─────────────────────────
def rm_anova_1way(data_2d):
    """data_2d: (n_subjects, n_conditions). Returns (F, df_b, df_e, p, eta2_p)."""
    n, k   = data_2d.shape
    grand  = np.nanmean(data_2d)
    row_m  = np.nanmean(data_2d, axis=1, keepdims=True)
    col_m  = np.nanmean(data_2d, axis=0, keepdims=True)
    ss_b   = n * np.sum((col_m - grand)**2)
    ss_s   = k * np.sum((row_m - grand)**2)
    ss_e   = np.sum((data_2d - grand)**2) - ss_b - ss_s
    df_b, df_e = k-1, (n-1)*(k-1)
    F  = (ss_b/df_b) / (ss_e/df_e)
    p  = stats.f.sf(F, df_b, df_e)
    return F, df_b, df_e, p, ss_b/(ss_b+ss_e)

# ── 3-SD outlier removal per level per measure ───────────────
def remove_3sd_outliers(arr):
    """arr: (n_sub, n_levels, n_meas). Matches MATLAB ana_taskPerformance.m."""
    out = arr.copy()
    _, n_lev, n_meas = out.shape
    for m in range(n_meas):
        for lv in range(n_lev):
            col = out[:, lv, m]
            mu, sd = np.nanmean(col), np.nanstd(col, ddof=1)
            if not np.isnan(mu) and sd > 0:
                out[(col < mu-3*sd) | (col > mu+3*sd), lv, m] = np.nan
        bad = np.isnan(out[:, :, m]).any(axis=1)
        out[bad, :, m] = np.nan
    return out

print("Statistical helper functions defined.")


In [ ]:
import matplotlib.pyplot as plt

## Load precomputed data

In [ ]:
sh_diff = np.load(os.path.join(OUT, 'shekhar_mle.npz'))['diff']   # (20,3,20)
r1_diff = np.load(os.path.join(OUT, 'rouault1_mle.npz'))['diff']  # (466,2,20)
r2_diff = np.load(os.path.join(OUT, 'rouault2_mle.npz'))['diff']  # (484,2,20)
print("Shekhar:", sh_diff.shape, "  Rouault1:", r1_diff.shape, "  Rouault2:", r2_diff.shape)


## 3-SD outlier removal

For each measure and difficulty level, flag values > 3 SD from the mean as NaN. Then propagate: if any level is NaN, set all levels to NaN.

In [ ]:
sh_clean = remove_3sd_outliers(sh_diff[:, [0,2], :])   # contrast 1 (hard) and 3 (easy)
r1_clean = remove_3sd_outliers(r1_diff)
r2_clean = remove_3sd_outliers(r2_diff)

for label, arr_raw, arr_cln in [
    ('Shekhar (contrasts 1&3)', sh_diff[:,[0,2],:], sh_clean),
    ('Rouault1', r1_diff, r1_clean),
    ('Rouault2', r2_diff, r2_clean),
]:
    n_removed = np.sum(np.isnan(arr_cln) & ~np.isnan(arr_raw))
    print(f"{label}: {n_removed} values set to NaN by 3SD removal")


## Supplementary Table 3 — Shekhar

In [ ]:
REPORTED_T3 = {"meta-d'":22.616,"AUC2":20.612,"Gamma":29.238,"Phi":10.898,
               "DeltaConf":14.834,"M-Ratio":-1.240,"AUC2-Ratio":-3.215,
               "M-Diff":-4.087,"AUC2-Diff":-4.006,"d'":23.777,"Criterion":0.166,"Confidence":14.543}

delta = sh_clean[:, 1, :] - sh_clean[:, 0, :]
print(f"{'Measure':<20} {'t (Python)':>12} {'t (MATLAB)':>12} {'match':>6}")
print("-"*55)
for m, name in enumerate(MEASURE_NAMES):
    t, df, p, d, lo, hi = ttest_1samp(delta[:, m])
    rep = REPORTED_T3.get(name, float('nan'))
    if np.isnan(t): print(f"{name:<20}         NaN {'---':>12}"); continue
    match = "✓" if abs(t-rep) < abs(rep)*0.05+0.05 else ("✗" if not np.isnan(rep) else "")
    print(f"{name:<20} {t:12.3f} {rep:12.3f}  {match}")


## Supplementary Table 4 — Rouault Expt 1

In [ ]:
REPORTED_T4 = {"meta-d'":35.285,"AUC2":35.405,"d'":49.278,"Confidence":32.390}
delta = r1_clean[:, 1, :] - r1_clean[:, 0, :]
print(f"{'Measure':<20} {'t (Python)':>12} {'t (MATLAB)':>12} {'match':>6}")
print("-"*55)
for m, name in enumerate(MEASURE_NAMES):
    t, df, p, d, lo, hi = ttest_1samp(delta[:, m])
    rep = REPORTED_T4.get(name, float('nan'))
    if np.isnan(t): print(f"{name:<20}         NaN {'---':>12}"); continue
    match = ("✓" if abs(t-rep)<abs(rep)*0.05+0.05 else "~") if not np.isnan(rep) else ""
    print(f"{name:<20} {t:12.3f} {rep if not np.isnan(rep) else float('nan'):12.3f}  {match}")


## Supplementary Table 5 — Rouault Expt 2

In [ ]:
REPORTED_T5 = {"meta-d'":15.304,"AUC2":13.657,"d'":48.583,"Confidence":15.334}
delta = r2_clean[:, 1, :] - r2_clean[:, 0, :]
print(f"{'Measure':<20} {'t (Python)':>12} {'t (MATLAB)':>12} {'match':>6}")
print("-"*55)
for m, name in enumerate(MEASURE_NAMES):
    t, df, p, d, lo, hi = ttest_1samp(delta[:, m])
    rep = REPORTED_T5.get(name, float('nan'))
    if np.isnan(t): print(f"{name:<20}         NaN {'---':>12}"); continue
    match = ("✓" if abs(t-rep)<abs(rep)*0.05+0.05 else "~") if not np.isnan(rep) else ""
    print(f"{name:<20} {t:12.3f} {rep if not np.isnan(rep) else float('nan'):12.3f}  {match}")


## Visualisation — effect of difficulty on each measure

In [ ]:
FIGS = os.path.join(REPO, 'notebooks', 'figures')
os.makedirs(FIGS, exist_ok=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)
datasets = [
    ('Shekhar (n=20)', sh_clean, 'difficulty (hard→easy)'),
    ('Rouault1 (n=466)', r1_clean, 'contrast (low→high)'),
    ('Rouault2 (n=484)', r2_clean, 'contrast (low→high)'),
]
for ax, (label, arr, xl) in zip(axes, datasets):
    means = np.nanmean(arr, axis=0)   # (2, 20)
    sems  = np.nanstd(arr, axis=0, ddof=1) / np.sqrt(np.sum(~np.isnan(arr[:,0,:]), axis=0))
    x = np.arange(N_MEAS)
    ax.bar(x-0.2, means[0], 0.35, yerr=sems[0], label='Hard/Low', color='#d55e00', alpha=0.8, capsize=3)
    ax.bar(x+0.2, means[1], 0.35, yerr=sems[1], label='Easy/High', color='#0072b2', alpha=0.8, capsize=3)
    ax.set_xticks(x); ax.set_xticklabels(MEASURE_NAMES, rotation=90, fontsize=7)
    ax.set_title(label, fontweight='bold')
    ax.axhline(0, color='k', linewidth=0.5, linestyle='--')
    ax.legend(fontsize=8)
    ax.set_xlabel(xl)
    ax.set_ylabel('Mean measure ± SEM')
plt.suptitle('Supplementary Figure 2: Difficulty Dependence', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGS, 'fig_difficulty.png'), dpi=150, bbox_inches='tight')
print("Saved fig_difficulty.png")
plt.show()
